In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

## 1. Load and inspect raw datasets
Fetch Bank Marketing (id 222), Wine Quality (id 186), and Student Performance (id 320) from the UCI repository. Print each dataset's shape, target column, label distribution, and missing values to decide how to preprocess and binarize labels.

In [ ]:

bank = fetch_ucirepo(id=222)
wine = fetch_ucirepo(id=186)
student = fetch_ucirepo(id=320)

for name, d in {"bank": bank, "wine": wine, "student": student}.items():
    X, y = d.data.features, d.data.targets
    print(f"=== {name} ===")
    print("X shape:", X.shape)
    print("target columns:", y.columns.tolist())
    print(y.iloc[:, -1].value_counts().sort_index())
    missing = X.isna().sum()
    print("missing values:\n", missing[missing > 0], "\n")

=== bank ===
X shape: (45211, 16)
target columns: ['y']
y
no     39922
yes     5289
Name: count, dtype: int64
missing values:
 job            288
education     1857
contact      13020
poutcome     36959
dtype: int64 

=== wine ===
X shape: (6497, 11)
target columns: ['quality']
quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5
Name: count, dtype: int64
missing values:
 Series([], dtype: int64) 

=== student ===
X shape: (649, 30)
target columns: ['G1', 'G2', 'G3']
G3
0      15
1       1
5       1
6       3
7      10
8      35
9      35
10     97
11    104
12     72
13     82
14     63
15     49
16     36
17     29
18     15
19      2
Name: count, dtype: int64
missing values:
 Series([], dtype: int64) 



## 2. Preprocess and binarize labels
- **Bank:** label is yes/no → 1/0. Drop `duration` (only known after the call, so it leaks the outcome). Fill missing categorical values with "unknown". Stratified subsample to 10,000 rows to keep SVM/KNN tuning tractable (Caruana & Niculescu-Mizil used 5,000 training cases).
- **Wine:** quality ≥ 7 → 1 ("good"), else 0.
- **Student:** final grade G3 ≥ 10 → 1 (pass), else 0. Drop G1 and G2 since they nearly determine G3.

Summary table shows size, feature types, and positive rate for each dataset.

In [4]:
def prep_bank(d, n=10000, seed=0):
    X = d.data.features.drop(columns=["duration"]).copy()
    y = (d.data.targets["y"] == "yes").astype(int)
    cat = X.select_dtypes(include="object").columns
    X[cat] = X[cat].fillna("unknown")
    # stratified subsample so SVM/KNN grid searches stay fast
    X, _, y, _ = train_test_split(X, y, train_size=n, stratify=y, random_state=seed)
    return X.reset_index(drop=True), y.reset_index(drop=True)

def prep_wine(d):
    X = d.data.features.copy()
    y = (d.data.targets["quality"] >= 7).astype(int)
    return X, y

def prep_student(d):
    X = d.data.features.drop(columns=["G1", "G2"], errors="ignore").copy()
    y = (d.data.targets["G3"] >= 10).astype(int)   # pass/fail
    return X, y

data = {
    "bank": prep_bank(bank),
    "wine": prep_wine(wine),
    "student": prep_student(student),
}

summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": X.shape[0],
        "features": X.shape[1],
        "numeric": X.select_dtypes(exclude="object").shape[1],
        "categorical": X.select_dtypes(include="object").shape[1],
        "positive rate": round(y.mean(), 3),
    }
    for name, (X, y) in data.items()
])
summary

,dataset,rows,features,numeric,categorical,positive rate
0,bank,10000,15,6,9,0.117
1,wine,6497,11,11,0,0.197
2,student,649,30,13,17,0.846


## 3. Categorical features per dataset
Reference list of categorical columns. These are one-hot encoded inside the pipeline during the experiment loop.

In [5]:
pd.DataFrame([
    {"dataset": name,
     "categorical columns": ", ".join(X.select_dtypes(include="object").columns)}
    for name, (X, y) in data.items()
]).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

,dataset,categorical columns
0,bank,"job, marital, education, default, housing, loan, contact, month, poutcome"
1,wine,
2,student,"school, sex, address, famsize, Pstatus, Mjob, Fjob, reason, guardian, schoolsup, famsup, paid, activities, nursery, higher, internet, romantic"


In [6]:
def make_preprocessor(X, scale):
    cat = X.select_dtypes(include="object").columns.tolist()
    num = X.select_dtypes(exclude="object").columns.tolist()
    return ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])

# name: (classifier, scale features?, hyperparameter grid)
models = {
    "RF": (RandomForestClassifier(n_estimators=500, random_state=0), False,
           {"clf__max_features": [1, 2, 4, 6, 8, 10]}),
    "SVM": (SVC(kernel="rbf"), True,
            {"clf__C": [0.01, 0.1, 1, 10, 100],
             "clf__gamma": [0.001, 0.01, 0.1, 1]}),
    "KNN": (KNeighborsClassifier(), True,
            {"clf__n_neighbors": [1, 3, 5, 9, 15, 25, 35, 51],
             "clf__weights": ["uniform", "distance"]}),
}

In [7]:
partitions = [0.2, 0.5, 0.8]   # fraction used for training
n_trials = 3

results, cv_store = [], []

for dname, (X, y) in data.items():
    for mname, (clf, scale, grid) in models.items():
        for train_frac in partitions:
            for trial in range(n_trials):
                X_tr, X_te, y_tr, y_te = train_test_split(
                    X, y, train_size=train_frac, stratify=y, random_state=trial)

                pipe = Pipeline([("prep", make_preprocessor(X, scale)), ("clf", clf)])
                cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial)
                search = GridSearchCV(pipe, grid, cv=cv, scoring="accuracy",
                                      return_train_score=True, n_jobs=-1)
                search.fit(X_tr, y_tr)
                best = search.best_estimator_

                results.append({
                    "dataset": dname, "model": mname,
                    "train_frac": train_frac, "trial": trial,
                    "train_acc": accuracy_score(y_tr, best.predict(X_tr)),
                    "val_acc": search.best_score_,
                    "test_acc": accuracy_score(y_te, best.predict(X_te)),
                    "best_params": search.best_params_,
                })
                cv_store.append({"dataset": dname, "model": mname,
                                 "train_frac": train_frac, "trial": trial,
                                 "cv_results": search.cv_results_})
                print(f"{dname} | {mname} | {int(train_frac*100)}/{int((1-train_frac)*100)} "
                      f"| trial {trial} | test {results[-1]['test_acc']:.3f}")

results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)
results_df.head()

bank | RF | 20/80 | trial 0 | test 0.890
bank | RF | 20/80 | trial 1 | test 0.891
bank | RF | 20/80 | trial 2 | test 0.894
bank | RF | 50/50 | trial 0 | test 0.894
bank | RF | 50/50 | trial 1 | test 0.889
bank | RF | 50/50 | trial 2 | test 0.894
bank | RF | 80/19 | trial 0 | test 0.900
bank | RF | 80/19 | trial 1 | test 0.892
bank | RF | 80/19 | trial 2 | test 0.898
bank | SVM | 20/80 | trial 0 | test 0.894
bank | SVM | 20/80 | trial 1 | test 0.896
bank | SVM | 20/80 | trial 2 | test 0.895
bank | SVM | 50/50 | trial 0 | test 0.895
bank | SVM | 50/50 | trial 1 | test 0.893
bank | SVM | 50/50 | trial 2 | test 0.895
bank | SVM | 80/19 | trial 0 | test 0.897
bank | SVM | 80/19 | trial 1 | test 0.894
bank | SVM | 80/19 | trial 2 | test 0.896
bank | KNN | 20/80 | trial 0 | test 0.885
bank | KNN | 20/80 | trial 1 | test 0.884
bank | KNN | 20/80 | trial 2 | test 0.888
bank | KNN | 50/50 | trial 0 | test 0.889
bank | KNN | 50/50 | trial 1 | test 0.887
bank | KNN | 50/50 | trial 2 | test 0.886
b

,dataset,model,train_frac,trial,train_acc,val_acc,test_acc,best_params
0,bank,RF,0.2,0,1.0,0.9010,0.889750,{'clf__max_features': 10}
1,bank,RF,0.2,1,1.0,0.8925,0.891375,{'clf__max_features': 4}
2,bank,RF,0.2,2,1.0,0.8915,0.894500,{'clf__max_features': 8}
3,bank,RF,0.5,0,1.0,0.8946,0.894200,{'clf__max_features': 10}
4,bank,RF,0.5,1,1.0,0.9008,0.889400,{'clf__max_features': 6}
